In [ ]:
"""
fitters_test.ipynb

Tests the fitter object, meant to 
fit all models to a given session.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-11
Python Version: 3.11.14
"""

from sg.fitter import LVMFamily
from src.squiggs.neuron_viewer import NeuronViewer
from src.squiggs.renderers import FitRenderer
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
"""
TODO:
backburner
- what about adding ReLU to the taskvar model and removing block?
- plot rasters, sanity check the psths
"""

In [ ]:
# check if you can get back m2 z-score psths from dividing tuber
# ask joao about the masking of the problem via zscore

## Fit 

In [ ]:
# load data
from sg.data import load_data_from_npy

unit_spike_times, trial_data, session_data, regions = load_data_from_npy(
    subj_idx=0, sess_idx=5
)

In [ ]:
family = LVMFamily(
    trial_data=trial_data,
    spike_times=unit_spike_times,
    session_data=session_data,
    regions=regions,
    n_latents_mult=2,
    n_latents_addt=2,
    sanity_check=2,
    task_vars=["response", "rewarded", "response_prev", "rewarded_prev"],
    refit=False,
    norm_activity=True,
)
family.fit_all()
family.eval()

In [ ]:
renderer = FitRenderer(
    family.mod_offset,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
phase = np.pi / 3
phi = np.sqrt(2)
t = family.psths["DMS"].shape[1]

potato = 2 * np.sin(np.arange(t)) + 0.5 * np.cos(np.arange(t)) + 2.5

tuber = (
    np.sin(phi * np.arange(t) / 2 + phase) ** 2
    + 2
    * (0.7 * (np.cos(np.arange(t) / 2 + phase + (np.pi / 2)) + 1) ** 0.75)
    * 0.5
    * np.sin(np.arange(t) / 2) ** 3
    + 2
)

In [ ]:
plot_summary(family, model=family.mod_gain, potato=tuber, mode="gain")

In [ ]:
plt.figure()
plt.imshow(family.robs.T, aspect="auto")
plt.show()

In [ ]:
from sg.eval_models import plot_cweights_mult

plot_cweights_mult(family)